In [1]:
import pandas as pd
import sqlite3

In [ ]:
df = pd.read_csv("../data/data_jobs.csv")

In [3]:
#удаляем мусорную колонку
df = df.drop("Unnamed: 0", axis=1)

In [4]:
#подключение к бд
conn = sqlite3.connect("../data/jobs.db")

In [8]:
df["Salary Estimate"] = df["Salary Estimate"].str.replace("(Glassdoor est.)", "", regex=False)

df["Salary Estimate"] = df["Salary Estimate"].str.replace("$", "", regex=False)

df["Salary Estimate"] = df["Salary Estimate"].str.replace("K", "", regex=False)

df[["min_salary", "max_salary"]] = df["Salary Estimate"].str.split("-", expand=True)

df["min_salary"] = pd.to_numeric(df["min_salary"], errors="coerce")

df["max_salary"] = pd.to_numeric(df["max_salary"], errors="coerce")

df["avg_salary"] = (df["min_salary"] + df["max_salary"]) / 2

In [9]:
#загружаем таблицу SQL
df.to_sql("jobs", conn, if_exists="replace", index=False)

2253

In [6]:
query = "SELECT * FROM jobs LIMIT 5"

pd.read_sql(query, conn)

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Easy Apply
0,"Data Analyst, Center on Immigration and Justic...",$37K-$66K (Glassdoor est.),Are you eager to roll up your sleeves and harn...,3.2,Vera Institute of Justice\n3.2,"New York, NY","New York, NY",201 to 500 employees,1961,Nonprofit Organization,Social Assistance,Non-Profit,$100 to $500 million (USD),-1,True
1,Quality Data Analyst,$37K-$66K (Glassdoor est.),Overview\n\nProvides analytical and technical ...,3.8,Visiting Nurse Service of New York\n3.8,"New York, NY","New York, NY",10000+ employees,1893,Nonprofit Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1,-1
2,"Senior Data Analyst, Insights & Analytics Team...",$37K-$66K (Glassdoor est.),We’re looking for a Senior Data Analyst who ha...,3.4,Squarespace\n3.4,"New York, NY","New York, NY",1001 to 5000 employees,2003,Company - Private,Internet,Information Technology,Unknown / Non-Applicable,GoDaddy,-1
3,Data Analyst,$37K-$66K (Glassdoor est.),Requisition NumberRR-0001939\nRemote:Yes\nWe c...,4.1,Celerity\n4.1,"New York, NY","McLean, VA",201 to 500 employees,2002,Subsidiary or Business Segment,IT Services,Information Technology,$50 to $100 million (USD),-1,-1
4,Reporting Data Analyst,$37K-$66K (Glassdoor est.),ABOUT FANDUEL GROUP\n\nFanDuel Group is a worl...,3.9,FanDuel\n3.9,"New York, NY","New York, NY",501 to 1000 employees,2009,Company - Private,Sports & Recreation,"Arts, Entertainment & Recreation",$100 to $500 million (USD),DraftKings,True


In [10]:
#средняя зп
query = """
SELECT AVG(avg_salary) as average_salary
FROM jobs
"""

pd.read_sql(query, conn)

,average_salary
0,72.123002


In [14]:
#топ индустрий по зп
query = """
SELECT
    Industry,
    ROUND(AVG(avg_salary), 2) as avg_salary
FROM jobs
GROUP BY Industry
ORDER BY avg_salary DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,Industry,avg_salary
0,Drug & Health Stores,95.25
1,Education Training Services,92.83
2,Health Care Products Manufacturing,89.80
3,Sports & Recreation,88.17
4,Gambling,88.00
5,News Outlet,87.00
6,Transportation Equipment Manufacturing,85.00
7,Electrical & Electronic Manufacturing,84.67
8,Utilities,83.25
9,Biotech & Pharmaceuticals,83.11


In [12]:
#топ компаний по зп
query = """
SELECT
    [Company Name],
    ROUND(AVG(avg_salary), 2) as avg_salary
FROM jobs
GROUP BY [Company Name]
ORDER BY avg_salary DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,Company Name,avg_salary
0,Zipongo\n4.0,150.0
1,Xcutives.com Inc,150.0
2,Ursus\n4.4,150.0
3,Tesla Motors\n3.4,150.0
4,Risk Management Solutions (RMS)\n3.9,150.0
5,OSI Engineering\n4.5,150.0
6,Nuro\n4.4,150.0
7,Netflix\n3.9,150.0
8,Moveworks\n5.0,150.0
9,Logic Planet\n3.1,150.0


In [19]:
#чистим строки 
df["company_clean"] = df["Company Name"].str.split("\n").str[0]
df[["Company Name", "company_clean"]].head()

,Company Name,company_clean
0,Vera Institute of Justice\n3.2,Vera Institute of Justice
1,Visiting Nurse Service of New York\n3.8,Visiting Nurse Service of New York
2,Squarespace\n3.4,Squarespace
3,Celerity\n4.1,Celerity
4,FanDuel\n3.9,FanDuel


In [20]:
#обновляем таблицу
df.to_sql("jobs", conn, if_exists="replace", index=False)

2253

In [21]:
query = """
SELECT
    company_clean,
    ROUND(AVG(avg_salary), 2) as avg_salary
FROM jobs
GROUP BY company_clean
ORDER BY avg_salary DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,company_clean,avg_salary
0,Zipongo,150.0
1,Xcutives.com Inc,150.0
2,Ursus,150.0
3,Tesla Motors,150.0
4,Risk Management Solutions (RMS),150.0
5,OSI Engineering,150.0
6,Nuro,150.0
7,Netflix,150.0
8,Moveworks,150.0
9,Logic Planet,150.0
